# Contrôle qualité des numéros de téléphone — EG FINESS

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.tel_checker import analyser_telephones, marquer_doublons

pd.set_option("display.max_colwidth", 80)
print("Imports OK")

Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Chargement des EG FINESS

In [2]:
query_eg = """
    SELECT
        idstructure_stru,
        nmfinessej_stru,
        nmfinessetab_stru,
        raisonsociale_stru,
        cdcommune_stru,
        lbvoie_stru,
        telephone_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EG'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""

df_eg = pd.read_sql(query_eg, conn)
print(f"EG FINESS actifs chargés : {len(df_eg):,}")

EG FINESS actifs chargés : 104,777


## 3. Vue d'ensemble

In [3]:
total    = len(df_eg)
renseigne = df_eg["telephone_stru"].apply(lambda x: str(x).strip() not in ("", "nan", "None")).sum()
absent    = total - renseigne

print("─" * 52)
print("PANORAMA DES TÉLÉPHONES EG FINESS")
print("─" * 52)
print(f"  EG actifs total            : {total:>10,}")
print(f"  Téléphone renseigné        : {renseigne:>10,}  ({renseigne/total*100:.1f} %)")
print(f"  Téléphone absent           : {absent:>10,}  ({absent/total*100:.1f} %)")
print("─" * 52)

────────────────────────────────────────────────────
PANORAMA DES TÉLÉPHONES EG FINESS
────────────────────────────────────────────────────
  EG actifs total            :    104,777
  Téléphone renseigné        :     88,369  (84.3 %)
  Téléphone absent           :     16,408  (15.7 %)
────────────────────────────────────────────────────


## 4. Nettoyage, classification et doublons

In [ ]:
print("Nettoyage + classification + doublons...")
df = analyser_telephones(df_eg, col_tel="telephone_stru", col_commune="cdcommune_stru")
df = marquer_doublons(df, col_tel_clean="telephone_clean", col_id="idstructure_stru")
print(f"Analyse terminée — {len(df):,} EG traités.")
df[["raisonsociale_stru", "telephone_stru", "telephone_clean", "categorie",
    "motif", "zone_numero", "zone_attendue"]].head(15)

## 5. Distribution des catégories

In [5]:
ordre = ["MANQUANT", "ANOMALIE_CRITIQUE", "SURTAXE", "MOBILE",
         "INCOHERENCE_GEO", "DOUBLON", "VALIDE"]
labels = {
    "MANQUANT": "Manquant (absent / faux vide)",
    "ANOMALIE_CRITIQUE": "[1] Anomalie critique",
    "SURTAXE": "[2] Surtaxe (08 payant)",
    "MOBILE": "[2] Mobile (06/07)",
    "INCOHERENCE_GEO": "[2] Incohérence géographique",
    "DOUBLON": "[2] Doublon",
    "VALIDE": "[0] Valide",
}
vc = df["categorie"].value_counts()

print("─" * 60)
print("DISTRIBUTION DES CATÉGORIES")
print("─" * 60)
for cat in ordre:
    n = int(vc.get(cat, 0))
    print(f"  {labels[cat]:<34} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 60)

────────────────────────────────────────────────────────────
DISTRIBUTION DES CATÉGORIES
────────────────────────────────────────────────────────────
  Manquant (absent / faux vide)      :   16,408  ( 15.7 %)
  [1] Anomalie critique              :       10  (  0.0 %)
  [2] Surtaxe (08 payant)            :       97  (  0.1 %)
  [2] Mobile (06/07)                 :    2,327  (  2.2 %)
  [2] Incohérence géographique       :      114  (  0.1 %)
  [2] Doublon                        :   22,799  ( 21.8 %)
  [0] Valide                         :   63,022  ( 60.1 %)
────────────────────────────────────────────────────────────


## 6. Anomalies critiques — détail par motif

In [ ]:
df_crit = df[df["categorie"] == "ANOMALIE_CRITIQUE"].copy()
print(f"Total anomalies critiques : {len(df_crit):,}\n")

COLS = ["nmfinessetab_stru", "raisonsociale_stru", "telephone_stru",
        "telephone_clean", "motif"]
cols = [c for c in COLS if c in df_crit.columns]

for motif, groupe in df_crit.groupby("motif"):
    print(f"\n{'─'*60}")
    print(f"  {motif}  ({len(groupe):,} cas)")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

## 7. Signaux qualité — surtaxes, mobiles, incohérences géo, doublons

In [ ]:
for cat, titre in [("SURTAXE", "SURTAXES — 08 payant (signal fort)"),
                   ("MOBILE", "MOBILES — 06/07 (signal faible)"),
                   ("INCOHERENCE_GEO", "INCOHÉRENCES GÉO (informatif — portabilité depuis 2023)"),
                   ("DOUBLON", "DOUBLONS (informatif — standard mutualisé possible)")]:
    g = df[df["categorie"] == cat]
    print(f"\n{'─'*60}")
    print(f"  {titre}  ({len(g):,} cas)")
    print(f"{'─'*60}")
    if len(g) == 0:
        print("  aucun cas")
        continue
    c = [x for x in ["nmfinessetab_stru", "raisonsociale_stru", "telephone_clean",
                     "zone_numero", "zone_attendue", "motif"] if x in g.columns]
    print(g[c].head(5).to_string(index=False))

## 8. Synthèse finale

In [8]:
n_total = len(df)
n_manq  = (df["categorie"] == "MANQUANT").sum()
n_rens  = max(n_total - n_manq, 1)

def nb(cat): return int((df["categorie"] == cat).sum())

print("=" * 60)
print("SYNTHÈSE GLOBALE — CONTRÔLE TÉLÉPHONE EG FINESS")
print("=" * 60)
print(f"  EG actifs analysés         : {n_total:>8,}")
print(f"  Téléphone absent           : {n_manq:>8,}  ({n_manq/n_total*100:.1f} %)")
print()
print(f"  ── Sur les {n_rens:,} numéros renseignés ──")
print(f"  [1] Anomalies critiques    : {nb('ANOMALIE_CRITIQUE'):>8,}")
print(f"  [2] Surtaxes               : {nb('SURTAXE'):>8,}")
print(f"  [2] Mobiles                : {nb('MOBILE'):>8,}")
print(f"  [2] Incohérences géo       : {nb('INCOHERENCE_GEO'):>8,}")
print(f"  [2] Doublons               : {nb('DOUBLON'):>8,}")
print(f"  [0] Numéros valides        : {nb('VALIDE'):>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — CONTRÔLE TÉLÉPHONE EG FINESS
  EG actifs analysés         :  104,777
  Téléphone absent           :   16,408  (15.7 %)

  ── Sur les 88,369 numéros renseignés ──
  [1] Anomalies critiques    :       10
  [2] Surtaxes               :       97
  [2] Mobiles                :    2,327
  [2] Incohérences géo       :      114
  [2] Doublons               :   22,799
  [0] Numéros valides        :   63,022


## 9. Export Excel — Rapport de signalement

In [9]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/tel")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_telephones_eg.xlsx"

COLS_EXPORT = [
    "idstructure_stru", "nmfinessetab_stru", "nmfinessej_stru",
    "raisonsociale_stru", "cdcommune_stru", "departement", "lbvoie_stru",
    "telephone_stru", "telephone_clean",
    "categorie", "motif", "zone_numero", "zone_attendue", "nb_structures_meme_tel",
]
cols = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

FEUILLES = [
    ("Manquants",           "MANQUANT",          "616161", "EEEEEE"),
    ("Anomalies_Critiques", "ANOMALIE_CRITIQUE", "C62828", "F8CECC"),
    ("Surtaxes",            "SURTAXE",           "E65100", "FFE0B2"),
    ("Mobiles",             "MOBILE",            "F9A825", "FFF9C4"),
    ("Incoherences_Geo",    "INCOHERENCE_GEO",   "1565C0", "BBDEFB"),
    ("Doublons",            "DOUBLON",           "00838F", "B2EBF2"),
    ("Numeros_Valides",     "VALIDE",            "2E7D32", "EBF5EB"),
]

n_total = len(df); n_manq = (df["categorie"] == "MANQUANT").sum(); n_rens = max(n_total - n_manq, 1)
def nb(cat): return int((df["categorie"] == cat).sum())

df_synth = pd.DataFrame([
    {"Catégorie": "EG actifs total",        "Nombre": n_total,             "Part": "100 %"},
    {"Catégorie": "Téléphone absent",            "Nombre": int(n_manq),         "Part": f"{n_manq/n_total*100:.1f} %"},
    {"Catégorie": "—",                            "Nombre": "",                 "Part": ""},
    {"Catégorie": "[1] Anomalies critiques",     "Nombre": nb("ANOMALIE_CRITIQUE"), "Part": f"{nb('ANOMALIE_CRITIQUE')/n_rens*100:.1f} %"},
    {"Catégorie": "[2] Surtaxes",                "Nombre": nb("SURTAXE"),       "Part": f"{nb('SURTAXE')/n_rens*100:.1f} %"},
    {"Catégorie": "[2] Mobiles",                 "Nombre": nb("MOBILE"),        "Part": f"{nb('MOBILE')/n_rens*100:.1f} %"},
    {"Catégorie": "[2] Incohérences géo",        "Nombre": nb("INCOHERENCE_GEO"), "Part": f"{nb('INCOHERENCE_GEO')/n_rens*100:.1f} %"},
    {"Catégorie": "[2] Doublons",                "Nombre": nb("DOUBLON"),       "Part": f"{nb('DOUBLON')/n_rens*100:.1f} %"},
    {"Catégorie": "[0] Numéros valides",         "Nombre": nb("VALIDE"),        "Part": f"{nb('VALIDE')/n_rens*100:.1f} %"},
])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom, cat, ent, fond in FEUILLES:
        donnees = df[df["categorie"] == cat][cols]
        if len(donnees) > 0:
            donnees.to_excel(writer, sheet_name=nom, index=False)
            style_entete(writer.sheets[nom], ent)
            remplir(writer.sheets[nom], fond)
            auto_width(writer.sheets[nom])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print("\nFeuilles produites :")
for nom, cat, _, _ in FEUILLES:
    print(f"  {nom:<22} : {nb(cat):>8,}")

Export OK → /home/jovyan/work/signalement_data_contact/results/tel/signalement_telephones_eg.xlsx

Feuilles produites :
  Manquants              :   16,408
  Anomalies_Critiques    :       10
  Surtaxes               :       97
  Mobiles                :    2,327
  Incoherences_Geo       :      114
  Doublons               :   22,799
  Numeros_Valides        :   63,022
